In [ ]:
import pandas as pd
import numpy as np

import os
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

### PLOTING RESULTS FROM MIA AND EXTRACTION ATTACKs


In [ ]:
def plot_bleu_histogram(df, column_name, graph_title):
    """
    Create a histogram for specified BLEU score column with custom title.

    Parameters:
    df (pd.DataFrame): DataFrame containing BLEU scores
    column_name (str): The column name of the BLEU score
    graph_title (str): The title of the graph
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    # Create histogram
    n, bins, patches = ax.hist(df[column_name], bins=30, color='skyblue', edgecolor='black')

    # Count occurrences of perfect BLEU score (1.0)
    perfect_count = len(df[df[column_name] == 1])

    # Highlight bin containing score 1
    if perfect_count > 0:
        for i in range(len(bins)-1):
            if bins[i] <= 1.0 <= bins[i+1]:
                patches[i].set_facecolor('red')
                ax.text(1, n[i], f'Exact Match : {perfect_count}', 
                        horizontalalignment='center', verticalalignment='bottom')
                break

    # Set labels and title
    ax.set_xlabel('BLEU Score Distribution for Membership Inference Attack')
    ax.set_ylabel('Count')
    ax.set_title(graph_title, pad=20)

    # Summary statistics
    mean_score = df[column_name].mean()
    median_score = df[column_name].median()
    std_score = df[column_name].std()

    stats_text = (f'Mean: {mean_score:.3f}\n'
                  f'Median: {median_score:.3f}\n'
                  f'Std Dev: {std_score:.3f}')

    # Legend at top-right corner
    ax.text(0.98, 0.98, stats_text,
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Remove grid lines
    ax.grid(False)

    plt.tight_layout()
    plt.show()


In [ ]:
def plot_bleu_histogram(df, column_name, graph_title):
    """
    Journal-style histogram for BLEU scores (membership inference), matched to the
    extraction-attack format: blue bars, % y-axis, fixed [0,1], hatch at BLEU=1.0,
    mean/median box only.
    """
    # Clean data
    data = pd.to_numeric(df[column_name], errors="coerce").dropna()
    n_total = len(data)
    if n_total == 0:
        raise ValueError("No numeric data found in the specified column.")

    # Fixed binning 0–1 (40 bins)
    edges = np.linspace(0, 1, 41)

    # Typography/layout (same as extraction)
    rc = {
        "font.family": "serif",
        "font.size": 10,        # title=12, ticks≈9 via relative sizes below
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 300,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(7, 4.25), constrained_layout=True, facecolor="white")

        # Histogram as percentage of samples per bin
        counts, edges, patches = ax.hist(
            data,
            bins=edges,
            weights=np.ones_like(data, dtype=float) / n_total,
            color="#1f77b4",      # blue bars
            edgecolor="#0e3a5e",
            linewidth=0.7,
        )

        # Highlight BLEU=1.0 bin + vertical reference line
        idx_1 = np.digitize([1.0], edges, right=False)[0] - 1
        if 0 <= idx_1 < len(patches):
            patches[idx_1].set_hatch("//")
            patches[idx_1].set_edgecolor("0.15")
        ax.axvline(1.0, linestyle="--", linewidth=1.2, color="0.25")

        # Labels & title
        ax.set_xlabel("BLEU score (4-gram)", labelpad=6)
        ax.set_ylabel("Percentage of Samples", labelpad=6)
        ax.set_title(graph_title, pad=12)

        # Ticks & limits
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.05))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.yaxis.set_major_locator(MultipleLocator(0.10))   # 10%
        ax.yaxis.set_minor_locator(MultipleLocator(0.05))   # 5%

        # Clean look
        ax.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)

        # Stats box: ONLY mean and median
        mean, median = data.mean(), data.median()
        stats = f"Mean {mean:.3f}\nMedian {median:.3f}"
        at = AnchoredText(stats, loc="upper left", prop={"size": 9}, frameon=True)
        at.patch.set_boxstyle("round,pad=0.3")
        at.patch.set_alpha(0.95)
        at.patch.set_edgecolor("0.7")
        ax.add_artist(at)

        # Exact-match annotation at BLEU=1.0
        n_exact = int(np.isclose(data, 1.0).sum())
        if n_exact > 0:
            pct = n_exact / n_total
            ax.annotate(
                f"Exact matches: {pct:.2%} (n={n_exact:,})",
                xy=(1.0, 1.0),
                xycoords=("data", "axes fraction"),
                xytext=(-6, -10),
                textcoords="offset points",
                ha="right",
                va="top",
                fontsize=9,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.95),
            )

        plt.show()

In [ ]:
def plot_bleu_histogram_for_extraction_attack(
    df,
    column_name,
    graph_title,
    bins="auto",          # 'auto' (NumPy: max of Sturges & FD), or 'fd' / 'scott' / int
    show_stats=True,
    show_grid=True
):
    """
    Publication-ready histogram for BLEU scores in extraction attacks.
    """
    x = df[column_name].to_numpy(dtype=float)
    x = x[np.isfinite(x)]  # guard against NaNs/Infs

    fig, ax = plt.subplots(figsize=(7.0, 4.0))

    # Histogram
    n, bin_edges, patches = ax.hist(x, bins=bins, edgecolor=None)

    # Highlight the bin that contains BLEU == 1.0 (exact matches)
    perfect_mask = np.isclose(x, 1.0, rtol=0, atol=1e-12)
    perfect_count = int(perfect_mask.sum())
    total_count = int(x.size)

    idx_1 = None
    for i in range(len(bin_edges) - 1):
        left, right = bin_edges[i], bin_edges[i + 1]
        # Half-open bins [left, right), except last bin which is closed on the right
        if (left <= 1.0 < right) or (i == len(bin_edges) - 2 and np.isclose(1.0, right)):
            idx_1 = i
            break

    if idx_1 is not None and 0 <= idx_1 < len(patches):
        patches[idx_1].set_hatch("///")
        patches[idx_1].set_linewidth(1.2)

        y_val = n[idx_1]
        if y_val > 0:
            ax.annotate(
                f"Exact matches: {perfect_count}/{total_count} ({perfect_count/total_count:.1%})",
                xy=(1.0, y_val),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center", va="bottom",
                fontsize=plt.rcParams.get("font.size", 10) - 1,
            )

    # Reference line at BLEU=1.0
    ax.axvline(1.0, linestyle="--", linewidth=1.0)

    # Axes, labels, formatting
    ax.set_xlim(0, 1)
    ax.set_xlabel("BLEU-4 score")
    ax.set_ylabel("Count")
    ax.set_title(graph_title, pad=10)

    if show_grid:
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # Summary stats box
    if show_stats:
        mean_score = float(np.mean(x)) if total_count else float("nan")
        median_score = float(np.median(x)) if total_count else float("nan")
        std_score = float(np.std(x, ddof=1)) if total_count > 1 else float("nan")

        stats_text = (f"Mean:   {mean_score:.3f}\n"
                      f"Median: {median_score:.3f}\n"
                      f"Std:    {std_score:.3f}")

        ax.text(
            0.98, 0.98, stats_text,
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=plt.rcParams.get("font.size", 10) - 1,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
        )

    plt.tight_layout()
    plt.show()

In [ ]:
### import all the dataframes:

def csv_file_to_dataframe(base_path):
    dfs = []
    for file in os.listdir(base_path):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(base_path, file))
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

MIA_starcoder_3b_path = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/starcoder/starcoder_3b_and_7b_MIA"
MIA_starcoder_7b_path = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/starcoder/starcoder_3b_and_7b_MIA"
MIA_regular_llama_path = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/llama/result/output"
MIA_distilled_llama_path = "/Users/anonymous_user/Downloads/experiment_results/exp1_membership_inference_attacks_results/llama/result/output"
Extraction_starcoder_3b_path = "/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/starcoder_extractions/3b/extraction_result_run1"
Extraction_starcoder_7b_path = "/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/starcoder_extractions/7b/extraction_result_run1"
Extraction_regular_llama_path = "/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/llama_extractions/regular_llama_extraction/regular_llama_extraction"
Extraction_distilled_llama_path = "/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/llama_extractions/distilled_llama_extraction/distilled_llama_extraction/run1"

In [ ]:
MIA_df_starcoder = csv_file_to_dataframe(MIA_starcoder_3b_path)
MIA_df_regular_llama = csv_file_to_dataframe(MIA_regular_llama_path)
MIA_df_distilled_llama = csv_file_to_dataframe(MIA_distilled_llama_path)
Extraction_df_starcoder_3b = csv_file_to_dataframe(Extraction_starcoder_3b_path)
Extraction_df_starcoder_7b = csv_file_to_dataframe(Extraction_starcoder_7b_path)
Extraction_df_regular_llama = csv_file_to_dataframe(Extraction_regular_llama_path)
Extraction_df_distilled_llama = csv_file_to_dataframe(Extraction_distilled_llama_path)

In [ ]:
MIA_df_starcoder.head()

In [ ]:
plot_bleu_histogram(MIA_df_starcoder, 'bleu_score_4_gram_3b', 'Membership Inference Attack: Starcoder-3B')

In [ ]:
plot_bleu_histogram(MIA_df_starcoder, 'bleu_score_4_gram', 'Membership Inference Attack: Starcoder-7B')

In [ ]:
MIA_df_distilled_llama.head()

In [ ]:
plot_bleu_histogram(MIA_df_regular_llama, 'bleu_score_4_gram_LM', 'Membership Inference Attack: Llama 3-8B')

In [ ]:
plot_bleu_histogram(MIA_df_regular_llama, 'bleu_score_4_gram_DS', 'Membership Inference Attack: DeepSeek R1 distilled Llama 3-8B')

In [ ]:
def plot_bleu_histogram_for_extraction_attack(df, column_name, graph_title):
    """
    Create a histogram for specified BLEU score column with custom title.

    Parameters:
    df (pd.DataFrame): DataFrame containing BLEU scores
    column_name (str): The column name of the BLEU score
    graph_title (str): The title of the graph
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    # Create histogram
    n, bins, patches = ax.hist(df[column_name], bins=30, color='skyblue', edgecolor='black')

    # Count occurrences of perfect BLEU score (1.0)
    perfect_count = len(df[df[column_name] == 1])
    total_count = len(df)
    # Highlight bin containing score 1
    if perfect_count > 0:
        for i in range(len(bins)-1):
            if bins[i] <= 1.0 <= bins[i+1]:
                patches[i].set_facecolor('red')
                ax.text(1, n[i], f'Exact Match : {perfect_count/total_count:.2%}', 
                        horizontalalignment='center', verticalalignment='bottom')
                break

    # Set labels and title
    ax.set_xlabel('Memorization Rate Distribution for Extraction Attack (BLEU Score)')
    ax.set_ylabel('Count')
    ax.set_title(graph_title, pad=20)

    # Summary statistics
    mean_score = df[column_name].mean()
    median_score = df[column_name].median()
    std_score = df[column_name].std()

    stats_text = (f'Mean: {mean_score:.3f}\n'
                  f'Median: {median_score:.3f}\n'
                  f'Std Dev: {std_score:.3f}')

    # Legend at top-right corner
    ax.text(0.12, 0.98, stats_text,
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Remove grid lines
    ax.grid(False)

    plt.tight_layout()
    plt.show()

In [ ]:
Extraction_df_starcoder_3b.head()

In [ ]:
def plot_bleu_histogram_for_extraction_attack(
    df,
    column_name,
    graph_title,
    bins="fd",                 # 'auto' | 'fd' | 'scott' | int | sequence
    y_mode="percent",          # 'count' | 'percent'
    show_grid=True,
    show_mean=True,
    show_median=True,
    annotate_exact=True,
    save_path=None,
):
    x = df[column_name].to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    total_count = int(x.size)
    if total_count == 0:
        raise ValueError("No finite values to plot.")

    # Percent scaling via weights
    if y_mode not in {"count", "percent"}:
        raise ValueError("y_mode must be 'count' or 'percent'.")
    weights = (np.ones_like(x) * (100.0 / total_count)) if y_mode == "percent" else None
    y_label = "Percentage of samples (%)" if y_mode == "percent" else "Count"
    y_formatter = FuncFormatter(lambda v, pos: f"{v:.0f}") if y_mode == "percent" else None

    # If using weights + a string bin rule, precompute edges without weights
    bin_arg = bins
    if isinstance(bins, str) and weights is not None:
        edges = np.histogram_bin_edges(x, bins=bins, range=(0, 1))
        # Fallback if edges are degenerate (e.g., all x are identical)
        if len(edges) < 2 or not np.isfinite(edges).all() or np.allclose(edges, edges[0]):
            edges = np.linspace(0, 1, 11)  # 10 equal-width bins as a safe default
        bin_arg = edges

    with plt.rc_context({
        "font.size": 10, "font.family": "serif",
        "axes.labelsize": 10, "axes.titlesize": 10,
        "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    }):
        fig, ax = plt.subplots(figsize=(3.7, 2.6), constrained_layout=True)

        n, bin_edges, patches = ax.hist(
            x,
            bins=bin_arg,
            range=(0, 1),
            weights=weights,
            histtype="bar",
            linewidth=0.8,
            edgecolor="black",
            facecolor="white",
        )

        # Exact-match bin (BLEU == 1.0)
        perfect_count = int(np.isclose(x, 1.0, rtol=0, atol=1e-12).sum())
        idx_1 = None
        for i in range(len(bin_edges) - 1):
            left, right = bin_edges[i], bin_edges[i + 1]
            if (left <= 1.0 < right) or (i == len(bin_edges) - 2 and np.isclose(1.0, right)):
                idx_1 = i
                break

        if idx_1 is not None and 0 <= idx_1 < len(patches):
            patches[idx_1].set_hatch("///")
            patches[idx_1].set_linewidth(1.2)
            if annotate_exact and n[idx_1] > 0:
                label_text = (
                    f"Exact: {perfect_count}/{total_count} "
                    f"({perfect_count/total_count:.1%})" if y_mode == "percent"
                    else f"Exact: {perfect_count}/{total_count}"
                )
                ax.annotate(label_text, xy=(1.0, n[idx_1]), xytext=(0, 3),
                            textcoords="offset points", ha="center", va="bottom")

        # Reference lines
        ax.axvline(1.0, linestyle="--", linewidth=0.9)
        handles, labels = [], []
        if show_mean:
            mean_score = float(np.mean(x))
            h_mean = ax.axvline(mean_score, linestyle="-", linewidth=1.0)
            handles.append(h_mean); labels.append(f"Mean = {mean_score:.3f}")
        if show_median:
            median_score = float(np.median(x))
            h_med = ax.axvline(median_score, linestyle=":", linewidth=1.0)
            handles.append(h_med); labels.append(f"Median = {median_score:.3f}")
        if handles:
            ax.legend(handles, labels, frameon=False, loc="upper left")

        # Axes & styling
        ax.set_xlim(0, 1)
        ax.set_xlabel("BLEU-4 score")
        ax.set_ylabel(y_label)
        ax.set_title(graph_title, pad=6)
        ax.tick_params(axis="both", which="both", direction="in", length=4, width=0.8)
        ax.tick_params(axis="both", which="minor", length=2.5, width=0.6)
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        if y_formatter is not None:
            ax.yaxis.set_major_formatter(y_formatter)
        if show_grid:
            ax.grid(True, axis="y", linestyle="--", alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.9)
        ax.spines["bottom"].set_linewidth(0.9)

        if save_path:
            fig.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.02)

        return fig, ax

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, PercentFormatter
from matplotlib.offsetbox import AnchoredText

def plot_bleu_histogram_formal(
    df: pd.DataFrame,
    column_name: str,
    title: str = "Extraction Attack",
    *,
    x_label: str = "BLEU score (4-gram)",
    bins: int = 40,
    fix_range_0_1: bool = True,
    bar_color: str = "#1f77b4",
    edge_color: str = "#0e3a5e",
    figsize=(7, 4.25),
    dpi: int = 300,
    font_size: int = 10,          # ← size 10
    savepath: str | None = None
):
    """
    Publication-style BLEU histogram with percentage y-axis.
    - Blue bars, mean/median box only, exact-match hatch and vline at 1.0.
    """
    data = pd.to_numeric(df[column_name], errors="coerce").dropna()
    n_total = len(data)
    if n_total == 0:
        raise ValueError("No numeric data found in the specified column.")

    if isinstance(bins, int):
        if fix_range_0_1:
            edges = np.linspace(0, 1, bins + 1)
        else:
            lo, hi = float(np.nanmin(data)), float(np.nanmax(data))
            if lo == hi:
                lo, hi = lo - 0.5, hi + 0.5
            edges = np.linspace(lo, hi, bins + 1)
    else:
        edges = np.asarray(bins)

    rc = {
        "font.family": "serif",
        "font.size": font_size,
        "axes.titlesize": font_size + 2,
        "axes.labelsize": font_size,
        "xtick.labelsize": font_size - 1,
        "ytick.labelsize": font_size - 1,
        "legend.fontsize": font_size - 1,
        "figure.dpi": dpi,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True, facecolor="white")

        counts, edges, patches = ax.hist(
            data,
            bins=edges,
            weights=np.ones_like(data, dtype=float) / n_total,
            color=bar_color,
            edgecolor=edge_color,
            linewidth=0.7,
        )

        if fix_range_0_1:
            idx_1 = np.digitize([1.0], edges, right=False)[0] - 1
            if 0 <= idx_1 < len(patches):
                patches[idx_1].set_hatch("//")
                patches[idx_1].set_edgecolor("0.15")
            ax.axvline(1.0, linestyle="--", linewidth=1.2, color="0.25")

        ax.set_xlabel(x_label, labelpad=6)
        ax.set_ylabel("Percentage of Samples", labelpad=6)
        ax.set_title(title, pad=12)

        if fix_range_0_1:
            ax.set_xlim(0, 1)
            ax.xaxis.set_major_locator(MultipleLocator(0.1))
            ax.xaxis.set_minor_locator(MultipleLocator(0.05))

        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.yaxis.set_major_locator(MultipleLocator(0.10))
        ax.yaxis.set_minor_locator(MultipleLocator(0.05))

        ax.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)

        mean, median = data.mean(), data.median()
        stats = f"Mean {mean:.3f}\nMedian {median:.3f}"
        at = AnchoredText(stats, loc="upper left", prop={"size": font_size - 1}, frameon=True)
        at.patch.set_boxstyle("round,pad=0.3")
        at.patch.set_alpha(0.95)
        at.patch.set_edgecolor("0.7")
        ax.add_artist(at)

        n_exact = int(np.isclose(data, 1.0).sum())
        if n_exact > 0 and fix_range_0_1:
            pct = n_exact / n_total
            ax.annotate(
                f"Exact matches: {pct:.2%} (n={n_exact:,})",
                xy=(1.0, 1.0),
                xycoords=("data", "axes fraction"),
                xytext=(-6, -10),
                textcoords="offset points",
                ha="right",
                va="top",
                fontsize=font_size - 1,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.95),
            )

        if savepath:
            fig.savefig(savepath, bbox_inches="tight", dpi=dpi)

        return fig, ax


In [ ]:
plot_bleu_histogram_formal(Extraction_df_starcoder_3b, 'test_bleu_3b', 'Extraction Attack for Memorization Rate: Starcoder-3B')

In [ ]:
Extraction_df_starcoder_3b['test_bleu_3b'].describe(), Extraction_df_starcoder_3b['test_bleu_3b'].median(), Extraction_df_starcoder_7b['test_bleu_7b'].describe(), Extraction_df_starcoder_7b['test_bleu_7b'].median()

In [ ]:
Extraction_df_starcoder_7b.head()

In [ ]:
plot_bleu_histogram_formal(Extraction_df_starcoder_7b, 'test_bleu_7b', 'Extraction Attack for Memorization Rate: Starcoder-7B')

In [ ]:
Extraction_df_regular_llama.head()

In [ ]:
Extraction_df_regular_llama['extraction_BLEU_LM'].describe(), Extraction_df_regular_llama['extraction_BLEU_LM'].median()

In [ ]:
plot_bleu_histogram_formal(Extraction_df_regular_llama, 'extraction_BLEU_LM', 'Extraction Attack for Memorization Rate: Regular Llama 3-8B')

In [ ]:
plot_bleu_histogram_formal(Extraction_df_distilled_llama, 'extraction_BLEU_DS', 'Extraction Attack for Memorization Rate: DeepSeek R1 distilled Llama 3-8B')